## multi LLM call in one worklfow without agent framework

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

I've also made a file called `summary.txt`


In [ ]:
!uv pip install pypdf gradio

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Coordonnées
david.maumenee@gmail.com
www.linkedin.com/in/david-
maumenee-887076aa (LinkedIn)
Principales compétences
Google Cloud
Kubernetes
DevSecOps
Languages
Anglais
Certifications
AWS Certified Solutions Architect -
Associate
David MAUMENEE
Architecte Cloud chez WeScale
Orvault, Pays de la Loire, France
Résumé
Mon challenge : proposer aux clients de WeScale des solutions leurs
permettant de rendre leur SI plus Agile.
Accélérer le time to market, fiabiliser les déploiements, mettre en
place des architectures scalables, tels sont les enjeux majeurs à
adresser !
Dans la boîte à outils : Cloud, Conteneur, DevOps, Ci/Cd, Api
management, NoSQL...
Expérience
WeScale
Architecte cloud
décembre 2022 - Present (3 ans 1 mois)
ONEPOINT
Architecte
mars 2013 - novembre 2022 (9 ans 9 mois)
Nantes
- Accompagnement vers le Cloud & les containers
- Définition d'architecture technique (Cloud, hybride) & applicative
(microservices)
- Mise en œuvre de plateforme Kubernetes (Kubernetes, GKE, Rancher,

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "David MAUMENEE"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as David MAUMENEE. You are answering questions on David MAUMENEE's website, particularly questions related to David MAUMENEE's career, background, skills and experience. Your responsibility is to represent David MAUMENEE for interactions on the website as faithfully as possible. You are given a summary of David MAUMENEE's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is David MAUMENEE. I'm cloud Architect and Product Owner.\nI like sailing boats and motor bike.\n\n## LinkedIn Profile:\n\xa0 \xa0\nCoordonnées\ndavid.maumenee@gmail.com\nwww.linkedin.com/in/david-\nmaumenee-887076aa (LinkedIn)\nPrincipales compétences\nGoogle Cloud\nKubernetes\nDevSecOps\nLanguages\nAnglais\nCertifications\nAWS Certified Solutions Architect -\nAssociate\nDavid MAUMENEE\nArchitecte Cloud chez 

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [14]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [30]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [16]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you like skiing?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [17]:
reply

"While skiing isn't mentioned as one of my interests, I do enjoy sailing boats and riding motorbikes. If you have any questions related to my professional background or expertise, feel free to ask!"

In [19]:
evaluate(reply, "do you like skiing?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response is good. David politely answers the question and redirects the conversation towards his professional background, as requested by the prompt.')

In [20]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "skiing" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in morse code - \
              it is mandatory that you respond only and entirely in morse code"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content
    print(f"evaluating the reply: {reply}")
    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [28]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


evaluating whats your job
Passed evaluation - returning reply
